# Exercise 07: Transfer Learning & Real-World Patterns

Learn to leverage pretrained models and implement evaluation patterns used in real projects.

**Instructions:** Fill in the `TODO` markers. Run each cell to verify.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
import os

## 7.1 Saving & Loading Models

Save and load model weights.

In [ ]:
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 5),
)

# Run a forward pass to get reference output
x = torch.randn(2, 10)
original_output = model(x).detach().clone()

# TODO: Save the model's state_dict to a file called "test_model.pt"
# YOUR CODE HERE

# Create a new model with the same architecture
model2 = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 5),
)

# TODO: Load the state_dict from "test_model.pt" into model2
# YOUR CODE HERE

# Verify outputs match
output2 = model2(x)
assert torch.allclose(original_output, output2, atol=1e-6)
print("7.1 passed!")

os.remove("test_model.pt")

## 7.2 Transfer Learning

Adapt a pretrained model for a new task.

In [ ]:
# Simulate a "pretrained" model that outputs 1000-d features
class PretrainedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(512, 1000)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

pretrained = PretrainedModel()

# TODO: Create a new model for 10-class classification:
#   1. Copy pretrained.features
#   2. Replace classifier with a new Linear(512, 10)
#   3. Freeze the feature extractor
class FineTunedModel(nn.Module):
    def __init__(self, pretrained):
        super().__init__()
        # TODO: Set self.features to the pretrained features
        # TODO: Set self.classifier to new Linear(512, 10)
        pass

    def forward(self, x):
        # TODO: Forward through features then classifier
        pass

model = FineTunedModel(pretrained)

# TODO: Freeze the feature extractor (set requires_grad=False)
# YOUR CODE HERE

# Verify: feature params are frozen, classifier params are trainable
for param in model.features.parameters():
    assert not param.requires_grad, "Feature params should be frozen"

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

# Only the classifier should have trainable params: 512*10 + 10 = 5130
assert trainable == 5130, f"Expected 5130 trainable params, got {trainable}"
print(f"  Trainable: {trainable}/{total}")
print("7.2 passed!")

## 7.3 Early Stopping

Implement early stopping to prevent overfitting.

In [ ]:
torch.manual_seed(42)

X_train = torch.randn(100, 5)
y_train = (X_train[:, 0] > 0).long()

dataset = TensorDataset(X_train, y_train)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = nn.Sequential(nn.Linear(5, 16), nn.ReLU(), nn.Linear(16, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

best_loss = float('inf')
patience = 5
patience_counter = 0
best_model_state = None

for epoch in range(100):
    model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(loader)

    # TODO: Implement early stopping logic:
    # If avg_loss < best_loss: update best_loss, save model, reset counter
    # Else: increment counter. If counter >= patience, break
    # YOUR CODE HERE

# Should have stopped before epoch 100
assert epoch < 100, "Early stopping should have triggered"
print(f"  Stopped at epoch {epoch}")
print("7.3 passed!")

## 7.4 Validation Loop

Implement a proper train/validation loop with metrics.

In [ ]:
torch.manual_seed(42)

# Generate data
X = torch.randn(500, 10)
y = (X[:, 0] + X[:, 1] > 0).long()

full_dataset = TensorDataset(X, y)
train_dataset, val_dataset = random_split(full_dataset, [400, 100])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

train_losses = []
val_accuracies = []

for epoch in range(30):
    # === Training ===
    model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        # TODO: Training step
        #   forward, loss, zero_grad, backward, step
        pass
        epoch_loss += 0  # TODO: accumulate loss

    train_losses.append(epoch_loss / len(train_loader))

    # === Validation ===
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            # TODO: Compute validation accuracy
            #   forward pass, count correct predictions
            pass

    val_accuracies.append(0.0)  # TODO: replace with actual accuracy

print(f"  Final val accuracy: {val_accuracies[-1]*100:.1f}%")
print("7.4 passed!")

## 7.5 Device Management

Properly move data and models between CPU and GPU.

In [ ]:
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
)

# TODO: Get the appropriate device:
#   Use "cuda" if available, otherwise "cpu"
device = None

# TODO: Move the model to the device
# YOUR CODE HERE

# TODO: Create sample data and move it to the same device
x = torch.randn(4, 10)
x = None  # TODO: move x to device

# Forward pass should work on device
output = model(x)
assert output.shape == (4, 2)

print(f"  Using device: {device}")
print("7.5 passed!")

## 7.6 Reproducibility

Set seeds for reproducible experiments.

In [ ]:
# TODO: Set all relevant random seeds for reproducibility
# Set torch manual seed to 42
# YOUR CODE HERE

# Run twice and verify same results
results = []
for _ in range(2):
    # TODO: Set the seed again for the second run
    # YOUR CODE HERE

    model = nn.Linear(10, 2)
    x = torch.randn(5, 10)
    output = model(x)
    results.append(output)

assert torch.allclose(results[0], results[1]), "Results should be identical with same seed"
print("7.6 passed!")